# SQZAST NCA Training

This notebook trains a **SQZAST** (Stochastic Quenched-Zoning Active Spin Texture) Neural Cellular Automata model.

**Key difference from rotation-invariant NCA:**
- Channel 3 (theta) still acts as a local orientation field that rotates the gradient perception
- However, theta's dynamics are **fixed** by a specific CA rule (not learned)
- The theta update rule uses stochastic site selection and directional updates with a step function
- Theta is subject to the same random initialization and noise as other channels
- Only the other channels participate in the learned dynamics and loss function

**Theta update rule:**
1. Random binary mask selects sites to update (probability 0.5)
2. Each selected site picks a random direction m in {1,2,3,4} (N,S,E,W)
3. For m=1 (North): dtheta = theta_north - theta; theta += dt * dtheta * step(-sin(2πdtheta))
4. For m=2 (South): dtheta = theta_south - theta; theta += dt * dtheta * step(-sin(2πdtheta))
5. For m=3 (East):  dtheta = theta_east - theta;  theta += dt * dtheta * step(+sin(2πdtheta))
6. For m=4 (West):  dtheta = theta_west - theta;  theta += dt * dtheta * step(+sin(2πdtheta))

**Note:** Theta is represented in the range [0,1] which maps to [0, 2π] radians. The perception uses cos(2πθ) and sin(2πθ) for gradient rotation, so the SQZAST update rule uses sin(2πdθ) for consistency.

In [ ]:
# @title Imports and Notebook Utilities
import os
import io
import PIL.Image, PIL.ImageDraw
import base64
import zipfile
import json
import requests
import numpy as np
import matplotlib.pylab as pl
import glob

from IPython.display import Image, HTML, Markdown, clear_output, display
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

os.environ['FFMPEG_BINARY'] = 'ffmpeg'
import moviepy.editor as mvp
from moviepy.video.io.ffmpeg_writer import FFMPEG_VideoWriter


def imread(url, max_size=None, mode=None):
    if isinstance(url, str) and url.startswith(('http:', 'https:')):
        headers = {
            "User-Agent": "Requests in Colab/0.0 (https://colab.research.google.com/; no-reply@google.com) requests/0.0"
        }
        r = requests.get(url, headers=headers)
        f = io.BytesIO(r.content)
    else:
        f = url
    img = PIL.Image.open(f)
    if max_size is not None:
        img.thumbnail((max_size, max_size), PIL.Image.LANCZOS)
    if mode is not None:
        img = img.convert(mode)
    img = np.float32(img) / 255.0
    return img


def np2pil(a):
    if a.dtype in [np.float32, np.float64]:
        a = np.uint8(np.clip(a, 0, 1) * 255)
    return PIL.Image.fromarray(a)


def imwrite(f, a, fmt=None):
    a = np.asarray(a)
    if isinstance(f, str):
        fmt = f.rsplit('.', 1)[-1].lower()
        if fmt == 'jpg':
            fmt = 'jpeg'
        f = open(f, 'wb')
    np2pil(a).save(f, fmt, quality=95)


def imencode(a, fmt='jpeg'):
    a = np.asarray(a)
    if len(a.shape) == 3 and a.shape[-1] == 4:
        fmt = 'png'
    f = io.BytesIO()
    imwrite(f, a, fmt)
    return f.getvalue()


def im2url(a, fmt='jpeg'):
    encoded = imencode(a, fmt)
    base64_byte_string = base64.b64encode(encoded).decode('ascii')
    return 'data:image/' + fmt.upper() + ';base64,' + base64_byte_string


def imshow(a, fmt='jpeg', id=None):
    return display(Image(data=imencode(a, fmt)), display_id=id)


def grab_plot(close=True):
    """Return the current Matplotlib figure as an image"""
    fig = pl.gcf()
    fig.canvas.draw()
    img = np.array(fig.canvas.renderer._renderer)
    a = np.float32(img[..., 3:] / 255.0)
    img = np.uint8(255 * (1.0 - a) + img[..., :3] * a)  # alpha
    if close:
        pl.close()
    return img


def zoom(img, scale=4):
    img = np.repeat(img, scale, 0)
    img = np.repeat(img, scale, 1)
    return img


class VideoWriter:
    def __init__(self, filename='_autoplay.mp4', fps=30.0, **kw):
        self.writer = None
        self.params = dict(filename=filename, fps=fps, **kw)

    def add(self, img):
        img = np.asarray(img)
        if self.writer is None:
            h, w = img.shape[:2]
            self.writer = FFMPEG_VideoWriter(size=(w, h), **self.params)
        if img.dtype in [np.float32, np.float64]:
            img = np.uint8(img.clip(0, 1) * 255)
        if len(img.shape) == 2:
            img = np.repeat(img[..., None], 3, -1)
        self.writer.write_frame(img)

    def close(self):
        if self.writer:
            self.writer.close()

    def __enter__(self):
        return self

    def __exit__(self, *kw):
        self.close()
        if self.params['filename'] == '_autoplay.mp4':
            self.show()

    def show(self, **kw):
        self.close()
        fn = self.params['filename']
        display(mvp.ipython_display(fn, **kw))

!nvidia-smi -L

In [ ]:
import torch
import torchvision.models as models

torch.set_default_tensor_type('torch.cuda.FloatTensor')

In [ ]:
#@title Loss Function Selection and Definitions
import torch.nn.functional as F

#@markdown ### Loss Function Type
loss_type = "sliced_ot"  #@param ["sliced_ot", "relaxed_ot"]

print(f"Selected loss function: {loss_type}")

# Load VGG for feature extraction
vgg = models.vgg16(weights='IMAGENET1K_V1').features

# ============================================================================
# Sliced OT Loss (rotation-invariant version)
# ============================================================================
def calc_styles_vgg(imgs, vgg):
    style_layers = [1, 6, 11, 18, 25]
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    x = (imgs - mean) / std
    b, c, h, w = x.shape
    features = [x.reshape(b, c, h * w)]
    for i, layer in enumerate(vgg[:max(style_layers) + 1]):
        x = layer(x)
        if i in style_layers:
            b, c, h, w = x.shape
            features.append(x.reshape(b, c, h * w))
    return features

def project_sort(x, proj):
    return torch.einsum('bcn,cp->bpn', x, proj).sort()[0]

def ot_loss(source, target, proj_n=32):
    ch, n = source.shape[-2:]
    projs = F.normalize(torch.randn(ch, proj_n), dim=0)
    source_proj = project_sort(source, projs)
    target_proj = project_sort(target, projs)
    target_interp = F.interpolate(target_proj, n, mode='nearest')
    return (source_proj - target_interp).square().sum()

def create_sliced_ot_loss(vgg, target_styles):
    """Create Sliced OT loss with rotation invariance."""
    def loss_f(imgs):
        source_features = calc_styles_vgg(imgs, vgg)
        min_loss = None
        for target_features in target_styles:
            loss = sum(ot_loss(x, y) for x, y in zip(source_features, target_features))
            if min_loss is None:
                min_loss = loss
            else:
                min_loss = torch.minimum(min_loss, loss)
        return min_loss
    return loss_f

# ============================================================================
# Relaxed OT Loss (rotation-invariant version)
# ============================================================================
class RelaxedOTLoss(torch.nn.Module):
    """https://arxiv.org/abs/1904.12785"""
    def __init__(self, vgg, target_styles, n_samples=1024):
        super().__init__()
        self.n_samples = n_samples
        self.vgg = vgg
        self.target_styles = target_styles  # List of rotated target features

    def get_vgg_features(self, imgs):
        style_layers = [1, 6, 11, 18, 25]
        mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
        std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
        x = (imgs - mean) / std
        b, c, h, w = x.shape
        features = [x.reshape(b, c, h * w)]
        for i, layer in enumerate(self.vgg[:max(style_layers) + 1]):
            x = layer(x)
            if i in style_layers:
                b, c, h, w = x.shape
                features.append(x.reshape(b, c, h * w))
        return features

    @staticmethod
    def pairwise_distances_cos(x, y):
        x_norm = torch.norm(x, dim=2, keepdim=True)
        y_t = y.transpose(1, 2)
        y_norm = torch.norm(y_t, dim=1, keepdim=True)
        dist = 1. - torch.matmul(x, y_t) / (x_norm * y_norm + 1e-10)
        return dist

    @staticmethod
    def style_loss(x, y):
        pairwise_distance = RelaxedOTLoss.pairwise_distances_cos(x, y)
        m1, m1_inds = pairwise_distance.min(1)
        m2, m2_inds = pairwise_distance.min(2)
        remd = torch.max(m1.mean(dim=1), m2.mean(dim=1))
        return remd

    @staticmethod
    def moment_loss(x, y):
        mu_x, mu_y = torch.mean(x, 1, keepdim=True), torch.mean(y, 1, keepdim=True)
        mu_diff = torch.abs(mu_x - mu_y).mean(dim=(1, 2))
        x_c, y_c = x - mu_x, y - mu_y
        x_cov = torch.matmul(x_c.transpose(1, 2), x_c) / (x.shape[1] - 1)
        y_cov = torch.matmul(y_c.transpose(1, 2), y_c) / (y.shape[1] - 1)
        cov_diff = torch.abs(x_cov - y_cov).mean(dim=(1, 2))
        return mu_diff + cov_diff

    def forward(self, generated_image):
        generated_features = self.get_vgg_features(generated_image)
        
        # Compute loss for each rotation and take minimum
        min_loss = None
        for target_features in self.target_styles:
            loss = 0.0
            for x, y in zip(generated_features, target_features):
                b_x, c_x, n_x = x.shape
                b_y, c_y, n_y = y.shape
                
                n_samples = min(n_x, n_y, self.n_samples)
                
                indices_x = torch.argsort(torch.rand(b_x, 1, n_x, device=x.device), dim=-1)[..., :n_samples]
                x_sampled = x.gather(-1, indices_x.expand(b_x, c_x, n_samples))
                
                indices_y = torch.argsort(torch.rand(b_y, 1, n_y, device=y.device), dim=-1)[..., :n_samples]
                y_sampled = y.gather(-1, indices_y.expand(b_y, c_y, n_samples))
                
                x_sampled = x_sampled.transpose(1, 2)
                y_sampled = y_sampled.transpose(1, 2)
                
                loss += self.style_loss(x_sampled, y_sampled) + self.moment_loss(x_sampled, y_sampled)
            
            if min_loss is None:
                min_loss = loss.mean()
            else:
                min_loss = torch.minimum(min_loss, loss.mean())
        
        return min_loss

print(f"VGG loaded and {loss_type} loss function defined.")

In [ ]:
#@title Load Target Image
from google.colab import files
from scipy import ndimage

print("Upload target texture:")
uploaded = files.upload()
texture_path = list(uploaded.keys())[0]

target_img = imread(io.BytesIO(uploaded[texture_path]), max_size=128)
print(f"Target texture:")
imshow(target_img)

# Compute rotation-invariant loss targets (64 angles)
# This allows the model to match ANY rotation of the target texture
print("\nComputing rotation-invariant loss targets (64 angles)...")
target_styles = []
for r in np.linspace(0.0, 360, 65)[:-1] + 0.12345:
    img_rotated = ndimage.rotate(target_img, r, reshape=False, mode='wrap')
    img_rotated_torch = torch.tensor(img_rotated).permute(2, 0, 1).unsqueeze(0)
    with torch.no_grad():
        style_features = calc_styles_vgg(img_rotated_torch, vgg)
        target_styles.append(style_features)

print(f"Precomputed {len(target_styles)} rotated style targets")

# Create loss function with rotation invariance
if loss_type == "sliced_ot":
    loss_fn = create_sliced_ot_loss(vgg, target_styles)
    print(f"\n\u2713 Created rotation-invariant sliced_ot loss function")
elif loss_type == "relaxed_ot":
    loss_fn = RelaxedOTLoss(vgg, target_styles, n_samples=1024)
    print(f"\n\u2713 Created rotation-invariant relaxed_ot loss function")

In [ ]:
#@title SQZAST NCA Architecture

#@markdown ### Model Architecture
channel_n = 12  #@param {type: "integer"}

def depthwise_conv(x, filters):
    """filters: [filter_n, h, w]"""
    b, ch, h, w = x.shape
    y = x.reshape(b * ch, 1, h, w)
    y = torch.nn.functional.pad(y, [1, 1, 1, 1], "circular")
    y = torch.nn.functional.conv2d(y, filters[:, None])
    return y.reshape(b, -1, h, w)


def sqzast_theta_update(theta, dt=1.0):
    """
    Fixed SQZAST update rule for the theta (orientation) channel.
    
    1. Random binary mask selects sites to update (probability 0.5)
    2. Each selected site picks a random direction m in {1,2,3,4} (N,S,E,W)
    3. For m=1 (North): dtheta = theta_north - theta; theta += dt * dtheta * step(-sin(2*pi*dtheta))
    4. For m=2 (South): dtheta = theta_south - theta; theta += dt * dtheta * step(-sin(2*pi*dtheta))
    5. For m=3 (East):  dtheta = theta_east - theta;  theta += dt * dtheta * step(+sin(2*pi*dtheta))
    6. For m=4 (West):  dtheta = theta_west - theta;  theta += dt * dtheta * step(+sin(2*pi*dtheta))
    
    Note: theta is in [0,1] representing [0, 2*pi] radians, so we use sin(2*pi*dtheta) 
    to match the angular interpretation used in the perception rotation.
    
    Args:
        theta: [b, 1, h, w] tensor of theta values
        dt: time step
    
    Returns:
        Updated theta tensor
    """
    b, _, h, w = theta.shape
    device = theta.device
    
    # Generate random binary mask (probability 0.5 for each site)
    mask = (torch.rand(b, 1, h, w, device=device) > 0.5).float()
    
    # Generate random direction for each site: 1=N, 2=S, 3=E, 4=W
    directions = torch.randint(1, 5, (b, 1, h, w), device=device)
    
    # Get neighbors using circular (periodic) boundary conditions
    # North neighbor: roll by -1 along height axis (row above)
    theta_north = torch.roll(theta, shifts=-1, dims=2)
    # South neighbor: roll by +1 along height axis (row below)
    theta_south = torch.roll(theta, shifts=1, dims=2)
    # East neighbor: roll by +1 along width axis (column to the right)
    theta_east = torch.roll(theta, shifts=1, dims=3)
    # West neighbor: roll by -1 along width axis (column to the left)
    theta_west = torch.roll(theta, shifts=-1, dims=3)
    
    # Compute dtheta for each direction
    dtheta_north = theta_north - theta
    dtheta_south = theta_south - theta
    dtheta_east = theta_east - theta
    dtheta_west = theta_west - theta
    
    # step function: step(x) = 0 if x < 0, 1 otherwise
    # For N,S: use step(-sin(2*pi*dtheta))
    # For E,W: use step(+sin(2*pi*dtheta))
    # Note: theta is in [0,1] representing [0, 2pi] radians, so we need 2*pi*dtheta
    two_pi = 2 * np.pi
    step_north = (torch.sin(two_pi * dtheta_north) <= 0).float()  # step(-sin(2*pi*dtheta))
    step_south = (torch.sin(two_pi * dtheta_south) <= 0).float()  # step(-sin(2*pi*dtheta))
    step_east = (torch.sin(two_pi * dtheta_east) >= 0).float()    # step(+sin(2*pi*dtheta))
    step_west = (torch.sin(two_pi * dtheta_west) >= 0).float()    # step(+sin(2*pi*dtheta))
    
    # Compute updates for each direction
    update_north = dt * dtheta_north * step_north
    update_south = dt * dtheta_south * step_south
    update_east = dt * dtheta_east * step_east
    update_west = dt * dtheta_west * step_west
    
    # Select the update based on direction
    update = torch.where(directions == 1, update_north,
             torch.where(directions == 2, update_south,
             torch.where(directions == 3, update_east,
                         update_west)))  # directions == 4
    
    # Apply mask and update theta
    theta_new = theta + mask * update
    
    return theta_new


class SqzastNCA(torch.nn.Module):
    """
    SQZAST Neural Cellular Automata.
    
    Similar to Rotation-Invariant NCA, but:
    - Channel 3 (theta) has FIXED dynamics defined by the SQZAST update rule
    - The theta channel is NOT learned, only the other channels are
    - Theta still acts as a local orientation field that rotates gradient perception
    - The bare value of theta is zeroed in the perception input (only gradients reach the network)
    """
    def __init__(self, chn=12, fc_dim=96, noise_level=1.0, theta_channel=3):
        super().__init__()
        self.chn = chn
        self.theta_channel = theta_channel  # First hidden channel after RGB (0,1,2)
        self.register_buffer("noise_level", torch.tensor([noise_level]))
        
        # Note: perception produces 4 features per channel, but we only learn
        # updates for non-theta channels. However, we still need full perception
        # because theta affects the rotated gradients for all channels.
        self.w1 = torch.nn.Conv2d(chn * 4, fc_dim, 1, bias=True)
        self.w2 = torch.nn.Conv2d(fc_dim, chn, 1, bias=False)

        torch.nn.init.xavier_normal_(self.w1.weight, gain=0.2)
        torch.nn.init.zeros_(self.w2.weight)

        with torch.no_grad():
            ident = torch.tensor([[0.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 0.0]])
            sobel_x = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]])
            lap_x = torch.tensor([[0.5, 0.0, 0.5], [2.0, -6.0, 2.0], [0.5, 0.0, 0.5]])
            self.filters = torch.stack([ident, sobel_x, sobel_x.T, lap_x, lap_x.T])

    def perception(self, s, dx=1.0, dy=1.0):
        """
        Compute rotation-invariant perception.
        
        For each channel, outputs 4 features:
        - identity (zeroed for theta channel)
        - rotated gradient x: cos(2*pi*theta)*sobel_x + sin(2*pi*theta)*sobel_y
        - rotated gradient y: -sin(2*pi*theta)*sobel_x + cos(2*pi*theta)*sobel_y
        - laplacian (rotation-invariant)
        """
        # Apply all 5 filters via depthwise conv
        z = depthwise_conv(s, self.filters)  # [b, 5*chn, h, w]

        # Extract theta (channel 3) and compute rotation
        theta = s[:, self.theta_channel:self.theta_channel+1, :, :]  # [b, 1, h, w]
        cos_t = torch.cos(2 * np.pi * theta)  # [b, 1, h, w]
        sin_t = torch.sin(2 * np.pi * theta)  # [b, 1, h, w]

        # Extract filter outputs (interleaved: id, sx, sy, lx, ly for each channel)
        ident_all = z[:, 0::5]    # [b, chn, h, w]
        sobel_x_all = z[:, 1::5]  # [b, chn, h, w]
        sobel_y_all = z[:, 2::5]  # [b, chn, h, w]
        lap_x_all = z[:, 3::5]    # [b, chn, h, w]
        lap_y_all = z[:, 4::5]    # [b, chn, h, w]

        # Rotate gradients using theta
        rotated_x = cos_t * sobel_x_all + sin_t * sobel_y_all
        rotated_y = -sin_t * sobel_x_all + cos_t * sobel_y_all

        # Combine laplacians (rotation-invariant)
        laplacian = lap_x_all + lap_y_all

        # Zero out bare theta value (channel 3 identity)
        # Only theta gradients should reach the network
        ident_all = ident_all.clone()
        ident_all[:, self.theta_channel, :, :] = 0.0

        # Stack features: [identity, rotated_x, rotated_y, laplacian] for each channel
        # Result: [b, 4*chn, h, w]
        perception = torch.stack([ident_all, rotated_x, rotated_y, laplacian], dim=2)
        return perception.reshape(s.shape[0], -1, s.shape[2], s.shape[3])

    def forward(self, s, dx=1.0, dy=1.0, dt=1.0, noise=None):
        # Add noise to all channels (including theta)
        if noise is not None:
            # Support both scalar and per-batch noise
            if isinstance(noise, torch.Tensor) and noise.ndim >= 1:
                if noise.ndim == 1:
                    noise = noise.reshape(-1, 1, 1, 1)
            s = s + torch.randn_like(s) * noise
        
        # Compute perception and learned dynamics
        z = self.perception(s, dx, dy)
        delta_s = self.w2(torch.relu(self.w1(z)))
        
        # Apply learned dynamics to all channels
        s_new = s + delta_s * dt
        
        # Override theta channel with fixed SQZAST dynamics
        # First, revert theta to its pre-learned-update state
        theta_old = s[:, self.theta_channel:self.theta_channel+1, :, :]
        if noise is not None:
            # Account for noise that was added
            pass  # theta_old already includes noise from the s update above
        
        # Apply the fixed SQZAST update rule to theta
        theta_new = sqzast_theta_update(theta_old, dt=dt)
        
        # Replace theta channel in the output
        s_new = s_new.clone()
        s_new[:, self.theta_channel:self.theta_channel+1, :, :] = theta_new
        
        return s_new

    def seed(self, n, h=128, w=128, noise_level=None):
        """
        Create initial seed states.
        
        Args:
            n: Number of samples (or list of per-sample noise levels)
            h: Height
            w: Width
            noise_level: Noise level(s) for initial state
        """
        if isinstance(n, (list, np.ndarray, torch.Tensor)):
            noise_level = n
            n = len(noise_level)
        
        if noise_level is None:
            nl = self.noise_level.item()
            return (torch.rand(n, self.chn, h, w) - 0.5) * nl
        elif isinstance(noise_level, (int, float)):
            return (torch.rand(n, self.chn, h, w) - 0.5) * noise_level
        else:
            if isinstance(noise_level, np.ndarray):
                noise_level = torch.tensor(noise_level, dtype=torch.float32)
            noise_level = noise_level.reshape(-1, 1, 1, 1)
            return (torch.rand(n, self.chn, h, w) - 0.5) * noise_level


def to_rgb(s):
    return s[..., :3, :, :] + 0.5


def to_theta(s, theta_channel=3):
    """Extract theta channel and map to [0, 1] for visualization."""
    return (s[..., theta_channel:theta_channel+1, :, :] + 0.5).clamp(0, 1)


# Initialize model with selected channel count
model = SqzastNCA(chn=channel_n)
param_n = sum(p.numel() for p in model.parameters())
print(f'SqzastNCA with {channel_n} channels')
print(f'Parameter count: {param_n}')
print(f'Theta channel: 3 (first hidden channel after RGB)')
print(f'Theta dynamics: FIXED (SQZAST rule, not learned)')

In [ ]:
#@title Setup Training
import os
import glob
from google.colab import files

#@markdown ### Training Parameters
init_noise = 1.0  #@param {type: "number"}
runtime_noise = 0.0  #@param {type: "number"}

print(f"Init noise: {init_noise}")
print(f"Runtime noise: {runtime_noise}")

# Check for existing weight files
weight_files = glob.glob('sqzast_*.pt') + glob.glob('checkpoint_*.pt')
weight_files = list(set(weight_files))

if weight_files:
    print(f"\nFound {len(weight_files)} weight file(s):")
    for i, f in enumerate(weight_files):
        file_size_kb = os.path.getsize(f) / 1024
        print(f"  [{i}] {f} ({file_size_kb:.1f} KB)")
    choice = input("\nEnter number to load, 'u' to upload, or Enter for fresh start: ").strip()

    if choice == 'u':
        print("Please upload your weights file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        state_dict = torch.load(filename)
    elif choice.isdigit() and 0 <= int(choice) < len(weight_files):
        filename = weight_files[int(choice)]
        print(f'Loading weights from: "{filename}"')
        state_dict = torch.load(filename)
    else:
        state_dict = None
else:
    print("\nNo weight files found in Colab storage.")
    upload_choice = input("Upload a weights file? (y/n, default=n): ").strip().lower()
    if upload_choice == 'y':
        print("Please upload your weights file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        state_dict = torch.load(filename)
    else:
        state_dict = None

# Load weights into model
if state_dict is not None:
    # Handle both checkpoint format (with 'model_state_dict') and raw state_dict
    if 'model_state_dict' in state_dict:
        model_weights = state_dict['model_state_dict']
        start_iter = state_dict.get('iteration', 0) + 1
        loss_log = state_dict.get('loss_log', [])
    else:
        # Raw state_dict (from weights file)
        model_weights = {k: v for k, v in state_dict.items() if not k.startswith('_')}
        start_iter = 0
        loss_log = []
    model.load_state_dict(model_weights)
    print(f"Loaded weights (starting from iteration {start_iter})")
else:
    start_iter = 0
    loss_log = []
    print("Starting fresh training")

# Always initialize pool from random seeds
pool_size = 256
with torch.no_grad():
    pool = model.seed(pool_size, noise_level=init_noise)
print(f"Initialized pool with {pool_size} random seeds (noise level {init_noise})")

# Optimizer with adaptive LR (always fresh)
opt = torch.optim.Adam(model.parameters(), 1e-3, capturable=True)
lr_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt, mode='min', factor=0.3, patience=500,
    threshold=0.01, threshold_mode='rel', min_lr=1e-6
)

print(f"\nPool shape: {pool.shape}")

In [ ]:
#@title Training Loop {vertical-output: true}

num_iterations = 10000  #@param {type: "integer"}
batch_size = 4  #@param {type: "integer"}

try:
    for i in range(start_iter, start_iter + num_iterations):
        with torch.no_grad():
            batch_idx = np.random.choice(len(pool), batch_size, replace=False)
            s = pool[batch_idx]

            # Inject fresh seed periodically
            if i % 32 == 0:
                s[:1] = model.seed(1, noise_level=init_noise)

        # Run forward steps
        step_n = np.random.randint(32, 96)
        for k in range(step_n):
            s = model(s, noise=runtime_noise)

        # Compute overflow loss (excluding theta channel which has fixed dynamics)
        # Create mask to exclude theta channel
        theta_ch = model.theta_channel
        s_for_overflow = torch.cat([s[:, :theta_ch], s[:, theta_ch+1:]], dim=1)
        overflow_loss = (s_for_overflow - s_for_overflow.clamp(-1.0, 1.0)).abs().sum()
        
        rgb = to_rgb(s)
        loss = overflow_loss + loss_fn(rgb)

        # Backward and optimize
        with torch.no_grad():
            loss.backward()
            for p in model.parameters():
                p.grad /= (p.grad.norm() + 1e-8)
            opt.step()
            opt.zero_grad()
            lr_sched.step(loss)
            pool[batch_idx] = s.detach()
            loss_log.append(loss.item())

            # Display progress
            if i % 10 == 0:
                lr = opt.param_groups[0]['lr']
                display(Markdown(f"iter: {i}, loss: {loss.item():.2e}, lr: {lr:.2e}"), display_id='stats')

            # Visualize
            if i % 20 == 0:
                pl.figure(figsize=(14, 3))
                
                # Loss plot
                pl.subplot(1, 3, 1)
                pl.plot(loss_log, '.', alpha=0.1)
                pl.yscale('log')
                pl.title('Loss')
                pl.xlabel('Iteration')

                # RGB output
                pl.subplot(1, 3, 2)
                imgs = rgb.permute(0, 2, 3, 1).cpu().numpy()
                pl.imshow(np.hstack(imgs))
                pl.title('RGB Output')
                pl.axis('off')

                # Theta field visualization
                pl.subplot(1, 3, 3)
                theta_imgs = to_theta(s).permute(0, 2, 3, 1).cpu().numpy()
                # Map theta to HSV hue for visualization
                import matplotlib.colors as mcolors
                theta_hue = np.tile(theta_imgs, (1, 1, 1, 3))
                for b in range(theta_hue.shape[0]):
                    for y in range(theta_hue.shape[1]):
                        for x in range(theta_hue.shape[2]):
                            h = theta_imgs[b, y, x, 0]
                            theta_hue[b, y, x] = mcolors.hsv_to_rgb([h, 1.0, 1.0])
                pl.imshow(np.hstack(theta_hue))
                pl.title('Theta Field (SQZAST dynamics)')
                pl.axis('off')

                pl.tight_layout()
                imshow(grab_plot(), id='progress')

            # Save checkpoint (weights only, no pool)
            if i % 1000 == 0 and i > start_iter:
                checkpoint = {
                    'iteration': i,
                    'model_state_dict': model.state_dict(),
                    'loss_log': loss_log,
                    'init_noise': init_noise,
                    'runtime_noise': runtime_noise,
                }
                torch.save(checkpoint, f'sqzast_iter_{i}.pt')
                print(f"\nCheckpoint saved at iteration {i}")

except KeyboardInterrupt:
    print('\n\nTraining interrupted by user!')
    print(f'Saving checkpoint at iteration {i}...')
    checkpoint = {
        'iteration': i,
        'model_state_dict': model.state_dict(),
        'loss_log': loss_log,
        'init_noise': init_noise,
        'runtime_noise': runtime_noise,
    }
    torch.save(checkpoint, f'sqzast_interrupted_iter_{i}.pt')
    print(f'Checkpoint saved!')

print(f'\nTraining completed at iteration {i}')

In [ ]:
#@title Test Model
import matplotlib.colors as mcolors

print("Testing SQZAST NCA:\n")

with torch.no_grad():
    # Run model
    s = model.seed(1, noise_level=init_noise)
    for _ in range(64):
        s = model(s, noise=runtime_noise)
    
    # Display RGB output
    rgb = to_rgb(s)[0].permute(1, 2, 0).cpu().numpy()
    
    # Display theta field as hue
    theta = to_theta(s)[0, 0].cpu().numpy()  # [h, w]
    theta_hsv = np.zeros((*theta.shape, 3))
    theta_hsv[..., 0] = theta  # hue
    theta_hsv[..., 1] = 1.0    # saturation
    theta_hsv[..., 2] = 1.0    # value
    theta_rgb = mcolors.hsv_to_rgb(theta_hsv)

    pl.figure(figsize=(12, 4))
    
    pl.subplot(1, 3, 1)
    pl.imshow(target_img)
    pl.title('Target')
    pl.axis('off')
    
    pl.subplot(1, 3, 2)
    pl.imshow(np.clip(rgb, 0, 1))
    pl.title('Model Output (RGB)')
    pl.axis('off')
    
    pl.subplot(1, 3, 3)
    pl.imshow(theta_rgb)
    pl.title('Theta Field (SQZAST)')
    pl.axis('off')
    
    pl.tight_layout()
    imshow(grab_plot())

# Show multiple samples to see theta variation
print("\nMultiple samples (showing SQZAST theta dynamics):")
with torch.no_grad():
    samples = []
    thetas = []
    for _ in range(4):
        s = model.seed(1, noise_level=init_noise)
        for _ in range(64):
            s = model(s, noise=runtime_noise)
        samples.append(to_rgb(s)[0].permute(1, 2, 0).cpu().numpy())
        theta = to_theta(s)[0, 0].cpu().numpy()
        theta_hsv = np.zeros((*theta.shape, 3))
        theta_hsv[..., 0] = theta
        theta_hsv[..., 1] = 1.0
        theta_hsv[..., 2] = 1.0
        thetas.append(mcolors.hsv_to_rgb(theta_hsv))

    pl.figure(figsize=(16, 4))
    
    pl.subplot(2, 1, 1)
    pl.imshow(np.hstack([np.clip(s, 0, 1) for s in samples]))
    pl.title('RGB Outputs')
    pl.axis('off')
    
    pl.subplot(2, 1, 2)
    pl.imshow(np.hstack(thetas))
    pl.title('Theta Fields (SQZAST dynamics)')
    pl.axis('off')
    
    pl.tight_layout()
    imshow(grab_plot())

In [ ]:
#@title Save Model Weights

# Save for demo (weights only)
weights_file = 'sqzast_weights.pt'

# Include training params for conversion script
state_dict = model.state_dict()
state_dict['_training_params'] = {
    'model_type': 'sqzast_nca',
    'theta_channel': model.theta_channel,
    'init_noise': init_noise,
    'runtime_noise': runtime_noise,
    'loss_type': loss_type,
    'texture_path': texture_path,
    'channel_n': channel_n,
}

torch.save(state_dict, weights_file)
print(f"Saved: {weights_file}")
print(f"  Channels: {channel_n}")
print(f"  Model type: sqzast_nca")
print(f"  Theta channel: {model.theta_channel}")
print(f"  Theta dynamics: FIXED (SQZAST rule)")
print(f"  Loss type: {loss_type}")
print(f"  Init noise: {init_noise}")
print(f"  Runtime noise: {runtime_noise}")

# Download
from google.colab import files
files.download(weights_file)